# Chapter 33: Visual Front End Pipeline

<a href="../lite/lab/index.html?path=ch33_visual_frontend.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

A camera produces millions of pixels per frame. But your SLAM system does not need millions
of points. It needs 200 reliable, well tracked feature points. The visual front end's job is
to find those 200 points, track them across frames, and ruthlessly reject the ones that are wrong.

This chapter covers the core pipeline: **detect**, **match**, **track**, **reject**.

## 33.1 Feature Detection

Good features are **corners**: regions where the image changes in two directions.
We simulate feature detection by identifying high gradient points.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_features = 50
img_size = 100
n_corners = 15       # true corner features
n_edge = 20          # edge features (less reliable)
n_noise = 15         # noise features
# ──────────────────────────────────────────────────────────────────────────────

# Simulate feature quality: corners are best, edges OK, noise bad
corners = np.random.uniform(10, 90, (n_corners, 2))
edges = np.random.uniform(10, 90, (n_edge, 2))
noise_pts = np.random.uniform(10, 90, (n_noise, 2))

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(corners[:, 0], corners[:, 1], c='forestgreen', s=80, marker='o', label=f'corners ({n_corners})', zorder=5)
ax.scatter(edges[:, 0], edges[:, 1], c='orange', s=50, marker='s', label=f'edges ({n_edge})')
ax.scatter(noise_pts[:, 0], noise_pts[:, 1], c='tomato', s=30, marker='x', label=f'noise ({n_noise})')
ax.set_xlim(0, img_size); ax.set_ylim(0, img_size)
ax.set_title("Detected features by quality", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 33.2 Matching

Match features between two frames by comparing their **descriptors**.
A descriptor is a compact representation of the image patch around each feature.

## 33.3 Tracking

Instead of re-detecting features every frame, **track** them using optical flow.
This is faster and produces smoother trajectories.

## 33.4 Outlier Rejection

**RANSAC** is the standard tool for rejecting outlier matches. It fits a geometric
model (e.g., fundamental matrix) to random subsets and keeps the largest consistent set.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_inliers = 30
n_outliers = 12
true_translation = np.array([10, 5])     # pixel translation between frames
ransac_iters = 100
threshold = 5.0
# ──────────────────────────────────────────────────────────────────────────────

# Source features
src = np.random.uniform(50, 500, (n_inliers + n_outliers, 2))
dst = src.copy()
dst[:n_inliers] += true_translation + np.random.normal(0, 1, (n_inliers, 2))
dst[n_inliers:] = np.random.uniform(50, 500, (n_outliers, 2))

# RANSAC for translation model
best_inliers = []
for _ in range(ransac_iters):
    idx = np.random.randint(len(src))
    t_est = dst[idx] - src[idx]
    errors = np.linalg.norm(dst - src - t_est, axis=1)
    inlier_mask = errors < threshold
    if inlier_mask.sum() > len(best_inliers):
        best_inliers = np.where(inlier_mask)[0]
        best_t = t_est

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
for i in range(len(src)):
    color = 'gray'
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], color=color, alpha=0.3)
ax.scatter(src[:, 0], src[:, 1], c='steelblue', s=20, label='frame 1')
ax.scatter(dst[:, 0], dst[:, 1], c='tomato', s=20, label='frame 2')
ax.set_title("All matches (including outliers)", fontsize=12); ax.legend()

ax = axes[1]
for i in best_inliers:
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], 'forestgreen', alpha=0.5)
outlier_idx = np.setdiff1d(range(len(src)), best_inliers)
for i in outlier_idx:
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], 'tomato', alpha=0.3)
ax.scatter(src[:, 0], src[:, 1], c='steelblue', s=20)
ax.scatter(dst[:, 0], dst[:, 1], c='tomato', s=20)
ax.set_title(f"RANSAC: {len(best_inliers)} inliers, {len(outlier_idx)} outliers", fontsize=12)

plt.tight_layout()
plt.show()

print(f"True translation: {true_translation}")
print(f"RANSAC estimate:  {best_t}")

**Key observations:**
- The visual front end is the most fragile part of visual SLAM.
- **Feature quality** matters more than quantity. 200 good features beat 2000 bad ones.
- **RANSAC** is essential. Without it, a single outlier can corrupt the pose estimate.
- The front end determines what the back end can and cannot do.

---

## Exercises

### Exercise 33.1
Implement RANSAC for estimating a 2D rigid transform (rotation + translation) from point
correspondences with 30% outliers. Plot inliers in green and outliers in red.

In [ ]:
# Your code here